In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# DAY 19 - CELL 1
# Dataset V1 Frozen Split

import os
import json
import numpy as np

BASE_PATH = "/content/drive/MyDrive/Underwater-Image-Data-set-main"

input_files = []
target_files = []

for root, dirs, files in os.walk(BASE_PATH):

    for file in files:

        if file.endswith("_input.png"):

            input_path = os.path.join(root, file)
            target_name = file.replace("_input.png", "_target.png")
            target_path = os.path.join(root, target_name)

            if os.path.exists(target_path):
                input_files.append(input_path)
                target_files.append(target_path)

pairs = list(zip(input_files, target_files))

print("Complete Input-Target pairs found:", len(pairs))

split_file = os.path.join(
    BASE_PATH,
    "Dataset_V1_frozen_split.json"
)

if os.path.exists(split_file):

    with open(split_file, "r") as f:
        split_data = json.load(f)

    train_pairs = [tuple(x) for x in split_data["train"]]
    val_pairs = [tuple(x) for x in split_data["validation"]]
    test_pairs = [tuple(x) for x in split_data["test"]]

    print("\nExisting frozen Dataset V1 split loaded.")

else:

    np.random.seed(42)
    np.random.shuffle(pairs)

    total = len(pairs)

    train_end = int(0.70 * total)
    val_end = int(0.85 * total)

    train_pairs = pairs[:train_end]
    val_pairs = pairs[train_end:val_end]
    test_pairs = pairs[val_end:]

    split_data = {
        "train": train_pairs,
        "validation": val_pairs,
        "test": test_pairs
    }

    with open(split_file, "w") as f:
        json.dump(split_data, f, indent=2)

    print("\nDataset V1 split created and frozen.")


print("\n========== FROZEN DATASET V1 ==========")
print("Training pairs   :", len(train_pairs))
print("Validation pairs :", len(val_pairs))
print("Testing pairs    :", len(test_pairs))
print("Split file       :", split_file)

Complete Input-Target pairs found: 1231

Existing frozen Dataset V1 split loaded.

========== FROZEN DATASET V1 ==========
Training pairs   : 863
Validation pairs : 184
Testing pairs    : 186
Split file       : /content/drive/MyDrive/Underwater-Image-Data-set-main/Dataset_V1_frozen_split.json


In [ ]:
import os
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

class UnderwaterDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        input_path, target_path = self.pairs[index]

        input_image = Image.open(input_path).convert("RGB")
        target_image = Image.open(target_path).convert("RGB")

        input_image = input_image.resize((256, 256))
        target_image = target_image.resize((256, 256))

        input_image = np.array(input_image).astype(np.float32) / 255.0
        target_image = np.array(target_image).astype(np.float32) / 255.0

        input_image = torch.tensor(input_image).permute(2, 0, 1)
        target_image = torch.tensor(target_image).permute(2, 0, 1)

        return input_image, target_image


train_dataset = UnderwaterDataset(train_pairs)
val_dataset = UnderwaterDataset(val_pairs)
test_dataset = UnderwaterDataset(test_pairs)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

print("========== DATA LOADERS ==========")
print("Training samples   :", len(train_dataset))
print("Validation samples :", len(val_dataset))
print("Testing samples    :", len(test_dataset))
print("Batch size         :", 8)
print("Training batches   :", len(train_loader))
print("Validation batches :", len(val_loader))
print("Testing batches    :", len(test_loader))

========== DATA LOADERS ==========
Training samples   : 863
Validation samples : 184
Testing samples    : 186
Batch size         : 8
Training batches   : 108
Validation batches : 23
Testing batches    : 24


In [ ]:
# DAY 19 - CELL 3
print("========== BATCH CHECK ==========")

inputs, targets = next(iter(train_loader))

print("Input shape  :", inputs.shape)
print("Target shape :", targets.shape)
print("Input range  :", inputs.min().item(), "to", inputs.max().item())
print("Target range :", targets.min().item(), "to", targets.max().item())

========== BATCH CHECK ==========
Input shape  : torch.Size([8, 3, 256, 256])
Target shape : torch.Size([8, 3, 256, 256])
Input range  : 0.0 to 1.0
Target range : 0.0 to 1.0


In [ ]:
# DAY 19 - CELL 4

import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.down1 = DoubleConv(3, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.middle = DoubleConv(128, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv1 = DoubleConv(64, 32)

        self.final = nn.Conv2d(32, 3, 1)

    def forward(self, x):
        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        d3 = self.down3(p2)
        p3 = self.pool3(d3)

        m = self.middle(p3)

        u3 = self.up3(m)
        u3 = torch.cat([u3, d3], dim=1)
        u3 = self.conv3(u3)

        u2 = self.up2(u3)
        u2 = torch.cat([u2, d2], dim=1)
        u2 = self.conv2(u2)

        u1 = self.up1(u2)
        u1 = torch.cat([u1, d1], dim=1)
        u1 = self.conv1(u1)

        return torch.sigmoid(self.final(u1))


model_test = UNet().to(device)

print("U-Net created successfully.")

Device: cpu
U-Net created successfully.


In [ ]:
# DAY 19 - CELL 5

import copy
import pandas as pd

EPOCHS = 5

experiments = [
    {
        "name": "Experiment A",
        "learning_rate": 0.001
    },
    {
        "name": "Experiment B",
        "learning_rate": 0.0001
    }
]

results = []

print("========== CONTROLLED EXPERIMENT ==========")
print("Loss function: L1 Loss")
print("Optimizer: Adam")
print("Epochs:", EPOCHS)
print("Batch size: 8")
print("Experiments:", len(experiments))

for experiment in experiments:
    print(
        experiment["name"],
        "| Learning rate:",
        experiment["learning_rate"]
    )

========== CONTROLLED EXPERIMENT ==========
Loss function: L1 Loss
Optimizer: Adam
Epochs: 5
Batch size: 8
Experiments: 2
Experiment A | Learning rate: 0.001
Experiment B | Learning rate: 0.0001


In [ ]:
# DAY 19 - CELL 5

import copy

epochs = 5

experiment_settings = {
    "Experiment A": {
        "learning_rate": 0.001,
        "loss_function": "L1Loss",
        "optimizer": "Adam",
        "epochs": 5
    },
    "Experiment B": {
        "learning_rate": 0.0001,
        "loss_function": "L1Loss",
        "optimizer": "Adam",
        "epochs": 5
    }
}

print("========== DAY 19 EXPERIMENT SETTINGS ==========")

for experiment, settings in experiment_settings.items():
    print(experiment)
    print("Learning rate :", settings["learning_rate"])
    print("Loss function :", settings["loss_function"])
    print("Optimizer     :", settings["optimizer"])
    print("Epochs        :", settings["epochs"])
    print()

========== DAY 19 EXPERIMENT SETTINGS ==========
Experiment A
Learning rate : 0.001
Loss function : L1Loss
Optimizer     : Adam
Epochs        : 5

Experiment B
Learning rate : 0.0001
Loss function : L1Loss
Optimizer     : Adam
Epochs        : 5



In [ ]:
# DAY 19 - CELL 5

import copy

epochs = 5

experiment_settings = {
    "Experiment A": {
        "learning_rate": 0.001,
        "loss_function": "L1Loss",
        "optimizer": "Adam",
        "epochs": 5
    },
    "Experiment B": {
        "learning_rate": 0.0001,
        "loss_function": "L1Loss",
        "optimizer": "Adam",
        "epochs": 5
    }
}

print("========== DAY 19 EXPERIMENT SETTINGS ==========")

for experiment, settings in experiment_settings.items():
    print(experiment)
    print("Learning rate :", settings["learning_rate"])
    print("Loss function :", settings["loss_function"])
    print("Optimizer     :", settings["optimizer"])
    print("Epochs        :", settings["epochs"])
    print()